# 서울 도서관 도서 목록 전처리
`seoul_library_book_list_202604.csv` → `data/processed/seoul_books.csv`

In [1]:
import pandas as pd
import numpy as np

RAW_PATH = "../../data/raw/seoul_library_book_list_202604.csv"
OUT_PATH  = "../../data/processed/seoul_books.csv"

## 1. 로드 및 기본 확인

In [4]:
# utf-8-sig: BOM 문자 자동 처리
rawDF = pd.read_csv(RAW_PATH, encoding="utf-8-sig")

# 마지막 열이 빈 컬럼이면 제거
prepDF = rawDF.loc[:, ~rawDF.columns.str.fullmatch(r"Unnamed.*")]

print("shape:", prepDF.shape)
print("columns:", prepDF.columns.tolist())
prepDF.head(3)

shape: (484483, 13)
columns: ['번호', '도서명', '저자', '출판사', '발행년도', 'ISBN', '세트 ISBN', '부가기호', '권', '주제분류번호', '도서권수', '대출건수', '등록일자']


/tmp/ipykernel_114496/937712551.py:2: DtypeWarning: Columns (0: 발행년도, 1: ISBN, 2: 권, 3: 주제분류번호) have mixed types. Specify dtype option on import or set low_memory=False.
  rawDF = pd.read_csv(RAW_PATH, encoding="utf-8-sig")


,번호,도서명,저자,출판사,발행년도,ISBN,세트 ISBN,부가기호,권,주제분류번호,도서권수,대출건수,등록일자
0,1,"단종, 영월에서의 124일",이규희 지음,이지북,2026,9791124035573,NaN,NaN,4.0,813.8,1,0,2026-04-29
1,2,"나는, 매주 비행기타고 제주 카페에 간다 :두 번째 이야기 .[2]",송현희 지음,부크크(bookk),2025,9791112089502,NaN,NaN,2.0,594.019,1,0,2026-04-29
2,3,"열혈 세종대왕 .5 ,충녕대군 왕이 되다",박지연,아울북,2026,9791173571336,NaN,NaN,5.0,911.05,1,0,2026-04-29


In [5]:
# 결측값 현황
prepDF.isnull().sum().sort_values(ascending=False)

부가기호       484483
세트 ISBN    461229
권          394921
주제분류번호     126453
출판사         19540
저자           1326
발행년도          542
도서명             0
번호              0
ISBN            0
도서권수            0
대출건수            0
등록일자            0
dtype: int64

In [6]:
prepDF.dtypes

번호           int64
도서명            str
저자             str
출판사            str
발행년도        object
ISBN        object
세트 ISBN    float64
부가기호       float64
권           object
주제분류번호      object
도서권수         int64
대출건수         int64
등록일자           str
dtype: object

## 2. 컬럼 정리

In [8]:
prepDF = prepDF.rename(columns={
    "번호":        "num_id",
    "도서명":      "title",
    "저자":        "author",
    "출판사":      "publisher",
    "발행년도":    "publish_year",
    "ISBN":        "isbn",
    "세트 ISBN":   "isbn_set",
    "부가기호":    "addition_symbol",
    "권":          "volume",
    "주제분류번호": "class_no",
    "도서권수":    "book_count",
    "대출건수":    "loan_count",
    "등록일자":    "registered_date",
})

prepDF.head(2)

,num_id,title,author,publisher,publish_year,isbn,isbn_set,addition_symbol,volume,class_no,book_count,loan_count,registered_date
0,1,"단종, 영월에서의 124일",이규희 지음,이지북,2026,9791124035573,NaN,NaN,4.0,813.8,1,0,2026-04-29
1,2,"나는, 매주 비행기타고 제주 카페에 간다 :두 번째 이야기 .[2]",송현희 지음,부크크(bookk),2025,9791112089502,NaN,NaN,2.0,594.019,1,0,2026-04-29


## 3. 타입 변환

In [9]:
prepDF["publish_year"]    = pd.to_numeric(prepDF["publish_year"], errors="coerce").astype("Int64")
prepDF["loan_count"]      = pd.to_numeric(prepDF["loan_count"],   errors="coerce").fillna(0).astype(int)
prepDF["registered_date"] = pd.to_datetime(prepDF["registered_date"], errors="coerce")

prepDF.dtypes

num_id                      int64
title                         str
author                        str
publisher                     str
publish_year                Int64
isbn                       object
isbn_set                  float64
addition_symbol           float64
volume                     object
class_no                   object
book_count                  int64
loan_count                  int64
registered_date    datetime64[us]
dtype: object

## 4. 필터링

In [10]:
print("전처리 전:", len(prepDF))

# ISBN 없는 행 제거
tmpDF = prepDF[prepDF["isbn"].notna() & (prepDF["isbn"].astype(str).str.strip() != "")]
print("ISBN 없는 행 제거 후:", len(tmpDF))

# 제목 없는 행 제거
filteredDF = tmpDF[tmpDF["title"].notna() & (tmpDF["title"].astype(str).str.strip() != "")]
print("제목 없는 행 제거 후:", len(filteredDF))

전처리 전: 484483
ISBN 없는 행 제거 후: 484483
제목 없는 행 제거 후: 484483


## 5. 중복 처리

In [11]:
# ISBN 기준 중복 확인
dup_count = filteredDF.duplicated(subset="isbn").sum()
print(f"ISBN 중복 행: {dup_count}건")

# 중복 시 대출건수 합산 후 첫 번째 행 유지
loan_sum = filteredDF.groupby("isbn")["loan_count"].sum().rename("loan_count_total")
dupSumDF = filteredDF.drop_duplicates(subset="isbn", keep="first").merge(loan_sum, on="isbn")
dupSumDF["loan_count"] = dupSumDF["loan_count_total"]
dupSumDF = dupSumDF.drop(columns=["loan_count_total"])

print("중복 제거 후:", len(dupSumDF))
dupSumDF.head(3)

ISBN 중복 행: 14698건
중복 제거 후: 469785


,num_id,title,author,publisher,publish_year,isbn,isbn_set,addition_symbol,volume,class_no,book_count,loan_count,registered_date
0,1,"단종, 영월에서의 124일",이규희 지음,이지북,2026,9791124035573,NaN,NaN,4.0,813.8,1,0,2026-04-29
1,2,"나는, 매주 비행기타고 제주 카페에 간다 :두 번째 이야기 .[2]",송현희 지음,부크크(bookk),2025,9791112089502,NaN,NaN,2.0,594.019,1,0,2026-04-29
2,3,"열혈 세종대왕 .5 ,충녕대군 왕이 되다",박지연,아울북,2026,9791173571336,NaN,NaN,5.0,911.05,1,0,2026-04-29


In [15]:
df = dupSumDF[dupSumDF['publish_year'].astype(str).str.match(r'^\d{4}$') & 
        dupSumDF['publish_year'].between(2000, 2026)]

In [16]:
len(df)

452334

## 6. 저장

In [ ]:
# import os
# os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
# print(f"저장 완료 -> {OUT_PATH}")
# print(f"최종 행 수: {len(df)}")